# Extrator híbrido — MinerU + Chandra em 2 fases

O `mineru` e o `chandra-ocr` pedem versões diferentes de `transformers`. Em vez de
isolar por ambiente, isola-se **no tempo**: cada fase roda com a versão que precisa.

| | Fase 1 | Fase 2 |
|---|---|---|
| Faz | páginas, YOLO, texto (MinerU) | tabelas, escaneadas e figuras (Chandra) |
| Entre elas | `Restart session` + sobe o `transformers` | |

A Fase 1 grava um **checkpoint** por PDF. A Fase 2 lê o checkpoint — então dá para
reiniciar o runtime à vontade sem perder trabalho, e re-executar só o que faltou.

## Ordem
1. `FASE 1 — instalar` → `FASE 1 — rodar`
2. **Runtime → Restart session**
3. `FASE 2 — instalar` → `FASE 2 — rodar`


## Config (rode nas duas fases)

In [ ]:
# ═══════════════ CONFIG — rode esta célula nas DUAS fases ═══════════════
# Só stdlib aqui: fitz/PIL/bs4 sao importados nas fases, depois do pip install delas.
import os, re, gc, json, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR   = Path('/content/drive/MyDrive/Pdfextractor')
PDFS_DIR   = BASE_DIR / 'data' / 'PDFs concluídos'

# pasta SEPARADA: não mistura com a extração feita pelo Docling
EXPORT_DIR = BASE_DIR / 'data' / 'SITE_EXPORT_MINERU'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

DPI_WEB, HIGH_DPI, MAX_LADO, YOLO_CONF = 150, 230, 1500, 0.40
MAX_PAGINAS = None
TEXTO_MIN_CHARS = 120     # menos texto extraível que isso = página escaneada -> OCR

CLASSES_YOLO = {0:'title',1:'plain_text',2:'abandon',3:'figure',4:'figure_caption',
                5:'table',6:'table_caption',7:'table_footnote',8:'isolate_formula',9:'formula_caption'}
CLS_TEXTO = {0,1,4,6,9}
CLS_ROTA  = {3:'figura', 5:'tabela', 8:'formula'}

# ── metadados da API da Squad 01 (a Fase 2 usa no fim) ──
META = {}
_mp = BASE_DIR/'metadados_api.json'
if _mp.exists(): META = json.loads(_mp.read_text(encoding='utf-8'))
def meta_do(nome):
    a = META.get(nome, {}); g = lambda *ks: next((str(a[k]).strip() for k in ks if a.get(k)), "")
    return {"titulo":g("Título","Titulo","title"),"autores":g("Autor(es)"),"ano":g("Ano"),
            "periodico":g("Título do periódico","Editora"),"volume":g("Volume"),"doi":g("DOI") or "—",
            "categoria":g("CATEGORIA"),"tipo":g("Tipo de documento"),"palavras_chave":g("Palavras-chave")}

def achar(pfx):
    for p in sorted(x.name for x in PDFS_DIR.glob('*.pdf')):
        if p.lower().startswith(pfx.lower()): return p
    return None

# ═══════════════ O QUE PROCESSAR ═══════════════
# Teste em 1 PDF. Para rodar em lote depois, troque por:
#     TARGETS = [n for n in sorted(x.name for x in PDFS_DIR.glob('*.pdf'))
#                if not (EXPORT_DIR/_slug(n)/'layout.json').exists()][:10]
TARGETS = [t for t in [achar('Bernardi')] if t]
assert TARGETS, f"PDF do Bernardi não encontrado em {PDFS_DIR}"

CKPT = EXPORT_DIR/'_checkpoint'
CKPT.mkdir(parents=True, exist_ok=True)

print("meta carregada :", len(META), "registros")
print("saída          :", EXPORT_DIR)
print("checkpoint     :", CKPT)
print("nesta execução :", len(TARGETS), "PDF(s)")
for t in TARGETS: print("   -", t)


---
# FASE 1 — MinerU (texto) + YOLO (layout) + PyMuPDF (páginas)

In [ ]:
!pip install -q "mineru[core]" doclayout-yolo PyMuPDF beautifulsoup4 Pillow pandas
import torch
print("GPU:", torch.cuda.is_available())

In [ ]:
# ── FASE 1 ──────────────────────────────────────────────────────────────────
import subprocess, tempfile, gc, time
import fitz
from PIL import Image
from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10

_p = hf_hub_download('juliozhao/DocLayout-YOLO-DocStructBench',
                     'doclayout_yolo_docstructbench_imgsz1024.pt')
yolo = YOLOv10(_p)

def render_paginas(page):
    esc = HIGH_DPI/72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(esc, esc))
    im_hi = Image.frombytes('RGB', [pix.width, pix.height], pix.samples); del pix
    im_web = im_hi; w, h = im_hi.size
    if max(w, h) > MAX_LADO:
        s = MAX_LADO/max(w, h); im_web = im_hi.resize((int(w*s), int(h*s)))
    return im_hi, im_web, im_web.size

def detectar_regioes(img_path, w, h, conf=YOLO_CONF):
    det = yolo.predict(str(img_path), imgsz=1024, conf=conf, verbose=False)
    b = det[0].boxes
    if b is None or len(b) == 0: return []
    cls = b.cls.cpu().numpy().astype(int); xy = b.xyxy.cpu().numpy(); cf = b.conf.cpu().numpy()
    return [{'classe_id': int(k), 'tipo_rota': CLS_ROTA.get(int(k)),
             'bbox': [float(x0)/w, float(y0)/h, float(x1)/w, float(y1)/h], 'conf': float(v)}
            for k, (x0, y0, x1, y1), v in zip(cls, xy, cf)]

# --- MinerU ---
_MAP = {'text':'texto','title':'texto','list':'texto','index':'texto',
        'interline_equation':'formula','equation':'formula','table':'tabela','image':'figura'}

def _txt_bloco(b):
    p = [sp.get('content') or sp.get('text') or ''
         for ln in (b.get('lines') or []) for sp in (ln.get('spans') or [])]
    p = [x for x in p if x]
    if not p:
        p = [b[k] for k in ('text', 'content') if b.get(k)]
    return ' '.join(p).strip()

def rodar_mineru(pdf_path):
    saida = Path(tempfile.mkdtemp())
    r = subprocess.run(['mineru', '-p', str(pdf_path), '-o', str(saida)],
                       capture_output=True, text=True, timeout=3600)
    if r.returncode != 0:
        raise RuntimeError('MinerU: ' + (r.stderr or r.stdout)[-400:])
    mids = list(saida.rglob('*middle*.json'))
    if not mids:
        raise RuntimeError('MinerU não gerou middle.json em ' + str(saida))
    dados = json.loads(mids[0].read_text(encoding='utf-8'))
    por_pag = {}
    for pg in dados.get('pdf_info', []):
        pno = pg.get('page_idx', 0) + 1
        W, H = (pg.get('page_size') or [1, 1])[:2]; W, H = (W or 1), (H or 1)
        out = []
        for b in (pg.get('para_blocks') or pg.get('preproc_blocks') or []):
            tipo = _MAP.get(b.get('type'), 'texto')
            if tipo == 'figura': continue
            bb = b.get('bbox') or [0, 0, 0, 0]
            bn = [round(bb[0]/W,4), round(bb[1]/H,4), round(bb[2]/W,4), round(bb[3]/H,4)]
            if tipo == 'tabela':
                html = ''
                for sub in (b.get('blocks') or [b]):
                    for ln in (sub.get('lines') or []):
                        for sp in (ln.get('spans') or []):
                            if sp.get('html'): html = sp['html']
                if html:
                    out.append({'tipo':'tabela','bbox':bn,'html_tabela':html}); continue
            t = _txt_bloco(b)
            if t: out.append({'tipo': tipo, 'bbox': bn, 'md': t})
        por_pag[pno] = out
    return por_pag

def fase1(nome):
    slug = re.sub(r'[^a-z0-9]+','-',Path(nome).stem.lower()).strip('-')[:60]
    ck = CKPT/f'{slug}.json'
    if ck.exists():
        print(f"  pulado (checkpoint existe): {slug}"); return
    out = EXPORT_DIR/slug; (out/'pages').mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f"\n=== FASE 1 · {nome}")
    mineru_pgs = rodar_mineru(PDFS_DIR/nome)

    fdoc = fitz.open(str(PDFS_DIR/nome)); N = len(fdoc)
    lim = N if MAX_PAGINAS is None else min(N, MAX_PAGINAS)
    paginas = []
    for pi in range(lim):
        pno = pi+1; fpage = fdoc[pi]
        im_hi, im_web, (w, h) = render_paginas(fpage)
        im_web.save(out/f'pages/p{pno:03d}.jpg', quality=85)
        im_hi.save(out/f'pages/hi_{pno:03d}.jpg', quality=92)      # a Fase 2 usa esta
        texto_cru = fpage.get_text('text') or ''
        regioes = detectar_regioes(out/f'pages/p{pno:03d}.jpg', w, h)
        tem_texto = len(texto_cru.strip()) >= TEXTO_MIN_CHARS
        tem_tabela = any(r['tipo_rota'] == 'tabela' for r in regioes)
        rota = 'chandra' if (not tem_texto or tem_tabela) else 'mineru'
        paginas.append({'n': pno, 'img': f'pages/p{pno:03d}.jpg', 'img_hi': f'pages/hi_{pno:03d}.jpg',
                        'w': w, 'h': h, 'texto_cru': texto_cru,
                        'tipo': 'escaneada' if not tem_texto else 'organica',
                        'rota': rota, 'regioes': regioes,
                        'blocos_mineru': mineru_pgs.get(pno, [])})
        print(f"    p{pno:03d} [{rota:<7}] mineru={len(mineru_pgs.get(pno,[])):3d} yolo={len(regioes)}")
        gc.collect()
    fdoc.close()
    ck.write_text(json.dumps({'arquivo': nome, 'slug': slug, 'n_paginas': N,
                              'processadas': lim, 'tempo_fase1_s': round(time.time()-t0,1),
                              'paginas': paginas}, ensure_ascii=False), encoding='utf-8')
    print(f"  checkpoint salvo: {ck.name}")

for nome in TARGETS:
    try:
        fase1(nome)
    except Exception as e:
        print(f"    ERRO em {nome}: {e}")
print("\n✅ FASE 1 concluída. Agora: Runtime -> Restart session, e siga para a Fase 2.")


---
# FASE 2 — Chandra

**Reinicie o runtime antes desta fase** (`Runtime → Restart session`), rode a célula de
Config de novo e siga daqui. O `transformers` sobe agora, para o Chandra OCR 2 carregar.

In [ ]:
!pip install -q "chandra-ocr[hf]" beautifulsoup4 Pillow
# transformers e huggingface_hub JUNTOS: o Chandra 2 (qwen3_5) exige versões novas
# das duas, e a Fase 1 (MinerU) deixou o hub numa versão antiga.
!pip install -q --upgrade transformers huggingface_hub accelerate
print("Instalado.")
print("Se a próxima célula der ImportError de huggingface_hub:")
print("   Runtime -> Restart session, rode a CONFIG e siga da próxima célula")
print("   (não precisa reinstalar nem refazer a Fase 1 — o checkpoint está salvo).")

In [ ]:
# ── verificação do ambiente da Fase 2 ──
import torch, transformers, huggingface_hub
print("transformers   :", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("GPU            :", torch.cuda.is_available())

In [ ]:
# ── FASE 2: funções (Chandra, tabelas, ordenação, LaTeX) ────────────────────
import re, json, gc, time
from pathlib import Path
from PIL import Image
from bs4 import BeautifulSoup
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from chandra.model.hf import generate_hf
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown

_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("🔄 Chandra..."); _t = time.time()
chandra = AutoModelForImageTextToText.from_pretrained('datalab-to/chandra-ocr-2',
                                                      dtype=_dtype, device_map='auto')
chandra.eval(); chandra.processor = AutoProcessor.from_pretrained('datalab-to/chandra-ocr-2')
chandra.processor.tokenizer.padding_side = 'left'
chandra.generation_config.max_new_tokens = 4096
chandra.generation_config.do_sample = False
chandra.generation_config.repetition_penalty = 1.10
chandra.generation_config.no_repeat_ngram_size = 0
print(f"✅ Chandra {time.time()-_t:.0f}s")

# ── render + crop ──
# render_paginas: só na Fase 1

def _crop_hi(im_hi, bbox, m=14):
    W,H = im_hi.size; x0,y0,x1,y1 = bbox
    return im_hi.crop((max(0,int(x0*W)-m), max(0,int(y0*H)-m), min(W,int(x1*W)+m), min(H,int(y1*H)+m)))

# ── Chandra ──
def _anti_rep(t, lim=4):
    out, ant, rep = [], None, 0
    for l in (t or '').splitlines():
        if l.strip() and l == ant:
            rep += 1
            if rep >= lim: continue
        else: rep, ant = 0, l
        out.append(l)
    return "\n".join(out)

def chandra_md(pil):
    res = generate_hf([BatchInputItem(image=pil.convert('RGB'), prompt_type='ocr_layout')], chandra)[0]
    raw = getattr(res,'raw','') or getattr(res,'markdown','') or ''
    try: md_ = parse_markdown(raw)
    except Exception: md_ = raw
    torch.cuda.empty_cache()
    return _anti_rep(md_ or '')

def classificar_figura(md_txt):
    txt = md_txt or ''
    n_num = len(re.findall(r'\d', txt)); n_alpha = len(re.findall(r'[A-Za-zÀ-ÿ]', txt))
    kw = ('eixo','axis','fig','regress','dose','kg','ha','ph','cm','mg','%')
    tem = any(any(k in t for k in kw) for t in re.findall(r'\w+', txt.lower()))
    if n_alpha < 8 and n_num < 4: return 'foto'
    if n_num >= 8 and (tem or n_alpha >= 15): return 'grafico'
    return 'grafico' if n_num >= n_alpha*0.30 else 'foto'

def limpar_figura(mdk):
    """MD de figura p/ vetor: remove ![](img) fantasma, $ vazios/soltos e tabelas DUPLICADAS,
    preservando reações (LaTeX), descrição e a tabela de dados. Robusto p/ figuras compostas
    (diagramas de reação + gráfico). A descrição PT vem da LEGENDA (associada em exportar_pdf)."""
    if not mdk: return ""
    t = re.sub(r'!\[[^\]]*\]\([^)]*\)', '', mdk)          # remove ![alt](img)
    t = re.sub(r'\$\s*\$', ' ', t)                        # remove $ $ vazios
    _seen = set()
    def _dedup(m):
        k = re.sub(r'\s+', '', m.group())
        if k in _seen: return ''
        _seen.add(k); return m.group()
    t = re.sub(r'<table.*?</table>', _dedup, t, flags=re.S)   # tabelas duplicadas -> 1
    t = re.sub(r'[ \t]{2,}', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t).strip()
    return t

# ── YOLO ──
# detectar_regioes: só na Fase 1

def ordenar_regioes(regs, x_split=0.5, full_thr=0.62):
    cx=lambda r:(r['bbox'][0]+r['bbox'][2])/2; wd=lambda r:r['bbox'][2]-r['bbox'][0]
    regs=sorted(regs,key=lambda r:(round(r['bbox'][1],3),cx(r)))
    ordem,esq,dirr=[],[],[]
    def flush():
        ordem.extend(sorted(esq,key=lambda r:r['bbox'][1])); ordem.extend(sorted(dirr,key=lambda r:r['bbox'][1]))
        esq.clear(); dirr.clear()
    for r in regs:
        if wd(r)>=full_thr: flush(); ordem.append(r)
        elif cx(r)<x_split: esq.append(r)
        else: dirr.append(r)
    flush(); return ordem

# ── tabela: html -> heads/regs -> texto/resumo ──
def _regs_de_html(html):
    trs = BeautifulSoup(html or '', 'html.parser').find_all('tr')
    if not trs: return [], []
    heads = [c.get_text(strip=True) for c in trs[0].find_all(['th','td'])]
    regs = []
    for tr in trs[1:]:
        cels = [c.get_text(strip=True) for c in tr.find_all(['td','th'])]
        if any(cels): regs.append({(heads[k] if k<len(heads) else f'col{k}'):v for k,v in enumerate(cels)})
    return heads, regs

def tabela_texto(titulo, heads, regs):
    p=[titulo] if titulo else []; p.append("Tabela com colunas: "+", ".join(heads)+".")
    idc=heads[0] if heads else ""
    for r in regs:
        ident=r.get(idc,""); vals=[f"{k} = {r[k]}" for k in heads[1:] if str(r.get(k,'')).strip()]
        if ident and vals: p.append(f"{idc} {ident}: "+"; ".join(vals)+".")
    return "\n".join(p)

def tabela_resumo(titulo, heads, regs):
    idc=heads[0] if heads else ""; itens=", ".join(str(r.get(idc,"")) for r in regs[:8])
    return ((titulo+" ") if titulo else "")+f"Tabela com {len(regs)} linhas e {len(heads)} colunas ({', '.join(heads)}). {idc}: {itens}."

def _md_table(heads, regs):
    if not heads: return ""
    esc = lambda s: str(s).replace("|","\\|").replace("\n"," ").strip()
    linhas = ["| " + " | ".join(esc(h) for h in heads) + " |",
              "| " + " | ".join("---" for _ in heads) + " |"]
    for r in regs:
        linhas.append("| " + " | ".join(esc(r.get(k,"")) for k in heads) + " |")
    return "\n".join(linhas)

def bloco_tabela(html, bbox):
    heads, regs = _regs_de_html(html)
    res = tabela_resumo("", heads, regs) if heads else ""
    tbl = _md_table(heads, regs)
    # MD p/ vetor: resumo em linguagem natural (âncora semântica) + tabela markdown (LLM-friendly)
    md = "\n\n".join(x for x in [res, tbl] if x) or "[tabela]"
    return {'tipo':'tabela','bbox':[round(v,4) for v in bbox],
            'md':md,
            'tabela_colunas':heads,'tabela_html':html,'tabela_json':regs,'tabela_resumo':res}

_SUP={'0':'⁰','1':'¹','2':'²','3':'³','4':'⁴','5':'⁵','6':'⁶','7':'⁷','8':'⁸','9':'⁹','+':'⁺','-':'⁻','(':'⁽',')':'⁾','n':'ⁿ','i':'ⁱ'}
_SUB={'0':'₀','1':'₁','2':'₂','3':'₃','4':'₄','5':'₅','6':'₆','7':'₇','8':'₈','9':'₉','+':'₊','-':'₋','(':'₍',')':'₎'}
def _conv(s, mapa): return ''.join(mapa.get(c,c) for c in s)
_FORMQ=[('(NH4)2SO4','(NH₄)₂SO₄'),('(NH₄)2SO4','(NH₄)₂SO₄'),('K2SO4','K₂SO₄'),('H2SO4','H₂SO₄'),
        ('CaCl2','CaCl₂'),('P2O5','P₂O₅'),('K2O','K₂O'),('CaCO3','CaCO₃'),('KNO3','KNO₃'),
        ('NaNO3','NaNO₃'),('NH4','NH₄'),('NO3','NO₃'),('NO2','NO₂'),('SO4','SO₄'),('PO4','PO₄'),
        ('H2O','H₂O'),('CO2','CO₂')]
def formatar_ciencia(t):
    """Isótopos, unidades com expoente e fórmulas químicas ASCII -> unicode (para páginas Docling)."""
    if not t: return t
    t=re.sub(r'\b15\s?N(?![a-zç])','¹⁵N',t); t=re.sub(r'\b13\s?C(?![a-zç])','¹³C',t)
    t=re.sub(r'\b14\s?N(?![a-zç])','¹⁴N',t); t=re.sub(r'\b18\s?O(?![a-zç])','¹⁸O',t)
    t=re.sub(r'\b(kg|ha|g|mg|µg|L|mL|dm|cm|mol|m|t)\s?-([123])\b', lambda m:m.group(1)+_conv('-'+m.group(2),_SUP), t)
    for a,b in _FORMQ: t=t.replace(a,b)
    t=re.sub(r'\)(\d)([A-Z])', lambda m:')'+_conv(m.group(1),_SUB)+m.group(2), t)
    return t

def limpar_sup_sub(t):
    t = re.sub(r'<sup>(.*?)</sup>', lambda m: _conv(m.group(1), _SUP), t or '', flags=re.I|re.S)
    t = re.sub(r'<sub>(.*?)</sub>', lambda m: _conv(m.group(1), _SUB), t, flags=re.I|re.S)
    return formatar_ciencia(t)
def _abaixo(b):
    y = min(0.95, b[3]+0.004)
    return [b[0], y, b[2], min(0.99, y+0.035)]

# ── monta blocos de uma página roteada pro CHANDRA (usa data-bbox do RAW do Chandra) ──
def chandra_raw(pil):
    """Roda o Chandra e devolve o RAW (com <div data-bbox data-label>) + tamanho da imagem enviada."""
    im = pil.convert('RGB')
    if max(im.size) > 1800:
        s = 1800/max(im.size); im = im.resize((int(im.size[0]*s), int(im.size[1]*s)))
    res = generate_hf([BatchInputItem(image=im, prompt_type='ocr_layout')], chandra)[0]
    raw = getattr(res, 'raw', '') or ''
    torch.cuda.empty_cache()
    return raw, im.size

def _div_para_texto(div):
    inner = div.decode_contents()
    inner = limpar_sup_sub(inner)                       # <sup>/<sub> -> unicode (+ química)
    return BeautifulSoup(inner, 'html.parser').get_text(' ', strip=True)

def blocos_chandra(im_hi, regioes=None):
    """Cada <div data-bbox data-label> do Chandra vira um bloco. AUTO-CALIBRA a bbox.
    O YOLO (regioes) manda na classe tabela-vs-figura: onde o YOLO diz FIGURA, o Chandra
    NÃO transforma em tabela — o recorte vai pela rota de figura (Chandra define subclasse)."""
    raw, _ = chandra_raw(im_hi)
    divs = [d for d in BeautifulSoup(raw, 'html.parser').find_all('div') if d.get('data-bbox')]
    coords = []
    for d in divs:
        try:
            v = [float(x) for x in d['data-bbox'].split()][:4]
            coords.append(v if len(v) == 4 else None)
        except Exception:
            coords.append(None)
    allx = [x for c in coords if c for x in (c[0], c[2])]
    ally = [y for c in coords if c for y in (c[1], c[3])]
    MX = (min(allx) + max(allx)) if allx else 1.0   # margem esquerda ≈ direita (preserva margem)
    MY = (min(ally) + max(ally)) if ally else 1.0   # margem topo ≈ base
    MX = MX or 1.0; MY = MY or 1.0
    figs = [r['bbox'] for r in (regioes or []) if r.get('tipo_rota') == 'figura']  # YOLO manda em figura
    blocos = []
    for d, v in zip(divs, coords):
        if not v:
            continue
        bbox = [round(min(1, min(v[0],v[2])/MX), 4), round(min(1, min(v[1],v[3])/MY), 4),
                round(min(1, max(v[0],v[2])/MX), 4), round(min(1, max(v[1],v[3])/MY), 4)]
        if figs and _dentro(bbox, figs, frac=0.45):
            continue                                   # YOLO diz FIGURA -> não vira tabela/texto aqui
        label = d.get('data-label') or 'Text'
        tab = d.find('table')
        if tab is not None or 'Table' in label:
            blocos.append(bloco_tabela(str(tab) if tab is not None else '', bbox))
        elif 'Picture' in label or 'Figure' in label:
            desc = _div_para_texto(d)
            blocos.append({'tipo': (classificar_figura(desc) if desc else 'foto'), 'bbox': bbox, 'md': desc})
        else:
            md = _div_para_texto(d)
            if md.strip():
                blocos.append({'tipo': 'texto', 'bbox': bbox, 'md': md})
    # FIGURAS pelo YOLO: recorte + Chandra OCR do crop (captura equação DENTRO do gráfico) + subclasse
    for r in [r for r in (regioes or []) if r.get('tipo_rota') == 'figura']:
        mdk = chandra_md(_crop_hi(im_hi, r['bbox']))
        blocos.append({'tipo': classificar_figura(mdk), 'bbox': [round(v,4) for v in r['bbox']],
                       'md': limpar_figura(mdk), 'conf': round(r.get('conf', 0), 3)})
    blocos = ordenar_regioes(blocos)                   # ordem por coluna (com as figuras no lugar)
    return blocos

# (blocos_docling removido — a Fase 2 só junta MinerU + Chandra)
def _norm(t): return re.sub(r'\s+',' ',(t or '')).strip().lower()
def _dentro(bn, regs, frac=0.5):
    a=max(1e-9,(bn[2]-bn[0])*(bn[3]-bn[1]))
    for r in regs:
        ix0,iy0=max(bn[0],r[0]),max(bn[1],r[1]); ix1,iy1=min(bn[2],r[2]),min(bn[3],r[3])
        if max(0,ix1-ix0)*max(0,iy1-iy0)/a>=frac: return True
    return False

# ── MD p/ o vetor (document.md): super/subscritos unicode -> LaTeX (LOSSLESS, LLM-friendly) ──
_SUP_INV = {'⁰':'0','¹':'1','²':'2','³':'3','⁴':'4','⁵':'5','⁶':'6','⁷':'7','⁸':'8','⁹':'9','⁺':'+','⁻':'-','⁽':'(','⁾':')','ⁿ':'n','ⁱ':'i'}
_SUB_INV = {'₀':'0','₁':'1','₂':'2','₃':'3','₄':'4','₅':'5','₆':'6','₇':'7','₈':'8','₉':'9','₊':'+','₋':'-','₍':'(','₎':')'}
_RE_SUP = re.compile('[' + re.escape(''.join(_SUP_INV)) + ']+')
_RE_SUB = re.compile('[' + re.escape(''.join(_SUB_INV)) + ']+')
def para_llm(t):
    """MD p/ vetor: converte super/subscritos unicode -> LaTeX. LOSSLESS — preserva isótopos,
    expoentes e índices químicos numa forma que o LLM lê sem ambiguidade e nada se perde:
    ¹⁵N -> ^{15}N ; kg⁻¹ -> kg^{-1} ; (NH₄)₂SO₄ -> (NH_{4})_{2}SO_{4} ; x² -> x^{2}.
    O Legível mantém o unicode; isto é exclusivo do document.md (vetor)."""
    if not t: return t
    t = _RE_SUP.sub(lambda m: '^{' + ''.join(_SUP_INV[c] for c in m.group()) + '}', t)
    t = _RE_SUB.sub(lambda m: '_{' + ''.join(_SUB_INV[c] for c in m.group()) + '}', t)
    return t

print("✅ Funções prontas.")

In [ ]:
# ── FASE 2: junta tudo e escreve a saída final ──────────────────────────────
def _crop_pil(im, bbox, m=14):
    W, H = im.size; x0, y0, x1, y1 = bbox
    return im.crop((max(0,int(x0*W)-m), max(0,int(y0*H)-m),
                    min(W,int(x1*W)+m), min(H,int(y1*H)+m)))

def fase2(ck_path):
    d = json.loads(Path(ck_path).read_text(encoding='utf-8'))
    slug = d['slug']; out = EXPORT_DIR/slug
    if (out/'layout.json').exists():
        print(f"  pulado (já finalizado): {slug}"); return None
    t0 = time.time(); print(f"\n=== FASE 2 · {slug}")
    paginas = []
    for pg in d['paginas']:
        im_hi = Image.open(out/pg['img_hi'])
        regioes = pg['regioes']
        if pg['rota'] == 'chandra':
            blocos = blocos_chandra(im_hi, regioes)
        else:
            figs = [r['bbox'] for r in regioes if r.get('tipo_rota') == 'figura']
            blocos = []
            for b in pg['blocos_mineru']:
                if figs and b.get('bbox') and _dentro(b['bbox'], figs, frac=0.45):
                    continue
                if b['tipo'] == 'tabela' and b.get('html_tabela'):
                    blocos.append(bloco_tabela(b['html_tabela'], b['bbox']))
                elif b.get('md'):
                    blocos.append({'tipo': b['tipo'], 'bbox': b['bbox'],
                                   'md': limpar_sup_sub(b['md'])})
            for r in [r for r in regioes if r.get('tipo_rota') == 'figura']:
                mdk = chandra_md(_crop_pil(im_hi, r['bbox']))
                blocos.append({'tipo': classificar_figura(mdk),
                               'bbox': [round(v,4) for v in r['bbox']],
                               'md': limpar_figura(mdk), 'conf': round(r.get('conf',0),3)})
            blocos = ordenar_regioes(blocos)

        for j, b in enumerate(blocos): b['id'] = f"p{pg['n']}-b{j}"
        for b in blocos:
            if b['tipo'] in ('grafico','foto') and b.get('bbox'):
                (out/'figures').mkdir(exist_ok=True)
                _crop_pil(im_hi, b['bbox']).save(out/'figures'/f"{b['id']}.jpg", quality=85)
                b['fig'] = f"figures/{b['id']}.jpg"
        for j, b in enumerate(blocos):
            if b['tipo'] in ('grafico','foto'):
                for k in (j+1, j-1):
                    if (0 <= k < len(blocos) and blocos[k]['tipo'] == 'texto'
                            and re.match(r'\s*(figura|fig\.)\s*\d', blocos[k].get('md','') or '', re.I)):
                        b['caption'] = blocos[k]['md']
                        b['md'] = (b['caption'] + "\n\n" + (b['md'] or '')).strip(); break

        paginas.append({'n': pg['n'], 'img': pg['img'], 'w': pg['w'], 'h': pg['h'],
                        'tipo': pg['tipo'], 'rota': pg['rota'], 'texto_cru': pg['texto_cru'],
                        'chars': sum(len(b.get('md','')) for b in blocos),
                        'counts': {'tabelas': sum(1 for b in blocos if b['tipo']=='tabela'),
                                   'figuras': sum(1 for b in blocos if b['tipo'] in ('grafico','foto')),
                                   'formulas': sum(1 for b in blocos if b['tipo']=='formula')},
                        'blocos': blocos})
        c = paginas[-1]['counts']
        print(f"    p{pg['n']:03d} [{pg['rota']:<7}] blocos={len(blocos)} tab={c['tabelas']} fig={c['figuras']}")
        (out/pg['img_hi']).unlink(missing_ok=True)          # libera a imagem grande
        gc.collect(); torch.cuda.empty_cache()

    resumo = {'organicas': sum(1 for p in paginas if p['tipo']=='organica'),
              'escaneadas': sum(1 for p in paginas if p['tipo']=='escaneada'),
              'tabelas': sum(p['counts']['tabelas'] for p in paginas),
              'figuras': sum(p['counts']['figuras'] for p in paginas),
              'formulas': sum(p['counts']['formulas'] for p in paginas)}
    layout = {'arquivo': d['arquivo'], 'slug': slug, 'n_paginas': d['n_paginas'],
              'processadas': d['processadas'], 'meta': meta_do(d['arquivo']), 'resumo': resumo,
              'tempo_s': round(d.get('tempo_fase1_s',0) + time.time()-t0, 1),
              'pipeline': 'hibrido-2fases: MinerU + Chandra + DocLayout-YOLO',
              'paginas': paginas}
    (out/'layout.json').write_text(json.dumps(layout, ensure_ascii=False, indent=2), encoding='utf-8')
    linhas = [f"# {layout['meta'].get('titulo') or slug}", ""]
    for p in paginas:
        linhas.append(f"\n---\n\n## Página {p['n']} ({p['rota']})\n")
        for b in p['blocos']:
            linhas.append(para_llm(b['md'])); linhas.append("")
    (out/'document.md').write_text("\n".join(linhas), encoding='utf-8')
    print(f"  ✅ {slug} | {resumo}")
    return layout

IDX_PATH = EXPORT_DIR/'index.json'
INDEX = json.loads(IDX_PATH.read_text(encoding='utf-8')) if IDX_PATH.exists() else []
feitos = {e['slug'] for e in INDEX}
for ck in sorted(CKPT.glob('*.json')):
    try:
        lay = fase2(ck)
        if lay and lay['slug'] not in feitos:
            INDEX.append({'slug': lay['slug'], 'arquivo': lay['arquivo'],
                          'n_paginas': lay['n_paginas'], 'meta': lay['meta'],
                          'resumo': lay['resumo']})
            feitos.add(lay['slug'])
            IDX_PATH.write_text(json.dumps(INDEX, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as e:
        print(f"    ERRO em {ck.stem}: {e}")
print(f"\n🎉 FASE 2 concluída — {len(INDEX)} documento(s) em {EXPORT_DIR}")
